# Evaluar el RAG en IRIS

En el Lab 02 construiste un RAG con un vector store persistente en IRIS. Ahora comprobaremos que recupera la fuente correcta y que se abstiene cuando no existe evidencia. No llamaremos a OpenRouter: buscamos una prueba reproducible de la recuperación.

## ¿Qué es una evaluación RAG?

Es una suite de tests para la recuperación de contexto. Antes de que un LLM redacte una respuesta, comprobamos si el RAG trae la fuente correcta. Para cada pregunta guardamos una expectativa, por ejemplo `¿Qué garantía tiene un producto reacondicionado? → warranty-policy.md`, y obtenemos `PASS` o `FAIL` según la primera fuente recuperada.

También probamos la abstención: si una pregunta no tiene evidencia, el sistema debe reconocerlo y no inventar una cita. Estas pruebas permiten detectar regresiones al cambiar documentos, *chunking*, embeddings, umbral o vector store. La prosa del LLM es otro nivel de evaluación. Aquí nos centramos en que reciba el contexto correcto.

## 1. Preparar las piezas

`learning_ai_rag` vuelve a aportar piezas pequeñas y visibles: el adaptador de IRIS busca los vectores, `RAGPipeline` conserva evidencia y citas, y las funciones de evaluación comparan el resultado con la expectativa del caso.

In [ ]:
from pathlib import Path
import sys

from dotenv import load_dotenv

root = Path.cwd().resolve()
while not (root / 'packages').exists():
    root = root.parent

load_dotenv(root / '.env')
sys.path.insert(0, str(root / 'packages' / 'learning_ai_rag'))

from learning_ai_rag import RAGPipeline, SentenceTransformerEmbedder, evaluate_cases, load_rag_cases  # noqa: E402
from learning_ai_rag.evaluation import pass_rate  # noqa: E402
from learning_ai_rag.ingest import chunk_documents, load_markdown_documents  # noqa: E402
from learning_ai_rag.iris_vector import IrisConnectionSettings, IrisVectorStore  # noqa: E402


## 2. Conocer los casos

Cada caso indica una pregunta y la fuente esperada. Los casos sin fuente esperada prueban que el sistema se abstiene en vez de inventar una cita.

In [ ]:
cases = load_rag_cases(root / 'datos-demo' / 'evaluation' / 'rag-cases.json')
for case in cases:
    expected = case.get('expected_source', 'abstención')
    print(f"- {case['id']}: {case['question']} → {expected}")


## 3. Comprobar el estado del Lab 02

Esta comprobación es de sólo lectura: verifica que la tabla vectorial, el índice HNSW y el corpus existen. Si falla, vuelve al Lab 02. Este notebook no crea ni carga datos por su cuenta.

In [ ]:
store = IrisVectorStore.connect(IrisConnectionSettings.from_environment())
chunk_count = store.check_ready()
print(f'Corpus comprobado: {chunk_count} chunks en IRIS.')


## 4. Evaluar cada caso

El pipeline crea el embedding de cada pregunta y recupera desde IRIS. Para cada caso, la evaluación compara la primera fuente recuperada —o la abstención— con la expectativa versionada.

In [ ]:
embedder = SentenceTransformerEmbedder()
chunks = chunk_documents(load_markdown_documents(root / "datos-demo" / "knowledge"))
pipeline = RAGPipeline(store, embedder, context_chunks=chunks)
results = evaluate_cases(pipeline, cases)

for result in results:
    status = 'PASS' if result.passed else 'FAIL'
    print(f'{status} {result.case_id}')
    expected = result.expected_source or 'abstención'
    retrieved = result.retrieved_source or 'sin cita'
    print(f'  Esperado: {expected}')
    print(f'  Recuperado: {retrieved}')
    print(f'  Fuentes del contexto: {result.retrieved_sources}')
    print(f'  Reglas ausentes: {result.missing_terms}')
    print(f'  Abstención: {result.abstained}\n')


## 5. Leer la tasa sin engañarnos

Una tasa alta es una señal útil, no una garantía absoluta. Si aparece un `FAIL`, investiga primero sus chunks y citas: así sabrás si debes revisar los documentos, el chunking, el embedding o el umbral.

In [ ]:
passed = sum(result.passed for result in results)
print(f'Tasa de aciertos: {pass_rate(results):.0%} ({passed}/{len(results)})')


## Repetir desde terminal

Tras recorrer los casos, ejecuta la misma evaluación de forma compacta:

```bash
python labs/03-python-rag-evaluable/evaluar_rag.py
```